# [$\rm{[Github]}$](https://github.com/tarocity/taiwan-fama-french-factor-dashboard)台股 $\rm{Fama–French\;Factor}$ 計算說明

#### $\rm{周育呈}$ <e11559560532@gmail.com>

### 目錄
* [說明](#說明)
* [因子計算方式](#因子計算方式)
* [Fama–French Factor Calculation](#Fama–French-Factor-Calculation)
* [Momentum Factor Calculation](#Momentum-Factor-Calculation)
* [Tableau 資料前處理](#Tableau-資料前處理)

----------------
# 說明
- `說明`  
這篇文檔說明`台股 Fama–French Factor`的 Tableau Dashboard 資料是如何計算的，Tableau Dashboard 的資料是使用 MSSQL 完成計算的，這篇文檔會以 python 計算並說明計算方式，如要查看完整的資料流程，包含抓取 FinMind API 資料 → 匯入MSSQL → 清理清料 → 計算 Fama Factor，可到 Github 中查看 .py 和 .sql 檔，平常是使用 Spyder IDE 寫 python，只有在寫說明文檔時會來用 Jupyter Lab，所以此篇文檔只包含資料流程`最後一步: 計算 Fama Factor`的一部分，其餘的資料流程可在 Github 中查看。   
原本想把 FinMind 可抓的資料都抓下來存到 MSSQL，結果發現筆電的空間不夠了，因此只抓了計算 Fama Factor 需要的資料，資料量不多`gold layer`也沒設計 Data Model 了。
- `資料來源`  
Tableau Dashboard 的使用到的原始資料包括: 還原股價、公司市值、資產負債表、綜合損益表、現金流量表、上市下市公司基本資料、加權報酬指數、臺灣十年期國債債券收益率，其中國債收益率來自 [Investing.com](https://hk.investing.com/rates-bonds/taiwan-10-year-bond-yield)，公司基本資料來自[公開資訊觀測站](https://mops.twse.com.tw/mops/#/web/t51sb01)，其餘資料皆來自 [FinMind](https://finmind.github.io/)。    
最後 Tableau Dashboard 資料頻率以月報酬呈現，所有因子與投組分為每年再平衡 (Rebalance) 一次與四次，即使用季財報再平衡與使用年財報再平衡，此篇文檔只呈現`每季再平衡`的投組計算方式，再平衡的詳細方法可查看下方`因子計算方式`的說明。



-------------------  
# 因子計算方式
計算方式皆採 [Kenneth R. French 網站](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html)的方法，即 [Fama, E. F., & French, K. R. (2015). A five-factor asset pricing model. Journal of Financial Economics, 116(1), 1–22.](https://www.sciencedirect.com/science/article/abs/pii/S0304405X14002323) 中的內容，基於資料的侷限性計算方式稍微不同，以下分別說明：
* MKTPR (Market Premium)  
  加權指數報酬率減去無風險利率，無風險利率使用臺灣十年期國債債券收益率。
* HML (High Minus Low)  
  依照樣本公司的市值中位數分為 2 組 (s1,s2)，並依照帳面市值比 (Book to Market ratio) 的 30 與 70 百分位數分為 3 組 (bm1,bm2,bm3) ，共 2x3=6 組投資組合，以大公司價值股減去大公司成長股的報酬率計算溢酬 (premuim)，小公司同上，兩溢酬平均數即為HML，即：
  $$\tt{HML=\frac{(s2bm3 - s2bm1)+(s1bm3 - s1bm1)}{2}}$$
* RMW (Robust Minus Weak)  
  依照樣本公司的市值中位數分為 2 組 (s1,s2)，並依照營業獲利 (Operating Profitability) 的 30 與 70 百分位數分為 3 組 (op1,op2,op3)，共 2x3=6 組投資組合，以大公司高獲利減去大公司低獲利的報酬率計算溢酬 (premuim)，小公司同上，兩溢酬平均數即為RMW，即：
  $$\tt{RMW=\frac{(s2op3 - s2op1)+(s1op3 - s1op1)}{2}}$$
* CMA (Conservative Minus Aggressive)  
  依照樣本公司的市值中位數分為 2 組 (s1,s2)，並依照總資產成長率 (Investment) 的 30 與 70 百分位數分為3組 (inv1,inv2,inv3)，共 2x3=6 組投資組合，以大公司低投資減去大公司高投資的報酬率計算溢酬 (premuim)，小公司同上，兩溢酬平均數即為CMA，即：
  $$\tt{CMA=\frac{(s2inv1 - s2inv3)+(s1inv1 - s1inv3)}{2}}$$
* SMB (Small Minus Big)  
  SMB分為 3 因子模型 $\tt SMB_3$ 與 5 因子模型 $\tt SMB_5$，以各因子的小公司減去大公司的報酬率計算溢酬 (premuim)，所有溢酬平均數即為SMB，即：
  $$\tt{SMB_{(BM)}=\frac{(s1bm1-s2bm1)+(s1bm2-s2bm2)+(s1bm3-s2bm3)}{3}} $$
  $$\tt{SMB_{(OP)}=\frac{(s1op1-s2op1)+(s1op2-s2op2)+(s1op3-s2op3)}{3}} $$
  $$\tt{SMB_{(INV)}=\frac{(s1inv1-s2inv1)+(s1inv2-s2inv2)+(s1inv3-s2inv3)}{3}} $$
  其中 $\tt SMB_3$ 與 $\tt SMB_5$ 分別為：
  $$\tt{SMB_{3}=SMB_{(BM)}} $$
  $$\tt{SMB_{5}=\frac{SMB_{(BM)}+SMB_{(OP)}+SMB_{(INV)}}{3}} $$
  Tableau Dashboard 只呈現 $\tt\color{red}{SMB_{5}}$
  
`補充`
* 其中營業獲利 (Operating Profitability) 依照文獻定義為營收 (revenues) 減去銷貨成本 (cost of
goods sold) 減去營業費用 (selling, general, and administrative expenses) 再減去利息支出 (interest expense) 最後除以股東權益 (book equity)，而 FinMind 的利息支出在現金流量表，且 FinMind 現金流量表資料與公開資訊觀測站的資料一致，皆為累計財報非單季財報，因此先把利息支出轉為單季資料後再進行因子計算，這步驟在 MSSQL 中完成。

* 投資組合分組方式與文獻相同但調整投組再平衡時間，文獻採每年七月再平衡投組，台股資料改採每年四月，如是每季再平衡一次，同案按照每季財報公布時間:
    - 第一季（Q1）季報：每年的 05 月 15 日前 公布 → Q1 財報：當年 6、7、8 月報酬
    - 第二季（Q2）季報：每年的 08 月 14 日前 公布 → Q2 財報：當年 9、10、11 月報酬
    - 第三季（Q3）季報：每年的 11 月 14 日前 公布 → Q3 財報：當年 12、隔年 1、2、3 月報酬
    - 第四季（Q4）季報：隔年的 03 月 31 日前 公布 → Q4 財報：隔年 4、5 月報酬

    上述再平衡的時間點一定會有部分公司遲交財報導致部分的`look-ahead bias`，但我沒找到所有公司的歷史實際財報公布日期，沒辦法去試算不同時間點的 look-ahead bias 比例，所以直接以法定財報公佈時間做為再平衡日期的依據。

  
* 公司市值資料擷取的時間點與財報一致，如以年財報再平衡即取 12 月月底公司市值，如以季財報再平衡即取 3、6、9 和 12 月月底公司市值。

* 因子皆獨立分組，市值與因子變數採獨立排序分組再取交集，並採市值加權 (Value-Weighted) 計算投資組合報酬率，樣本為台股上市上櫃下市下櫃公司並剔除金融業，資料期間為 2012 年 6 月至 2026 年 5 月。

除了上述的 Fama 五因子，其他因子中最常見的就是動能因子，Fama 三因子加上動能因子就是 Carhart 四因子模型。  
* UMD (Up Minus Down)
  依照樣本公司的市值中位數分為 2 組 (s1,s2)，並依照報酬動能 (Momentum) 的 30 與 70 百分位數分為 3 組 (mom1,mom2,mom3)，共 2x3=6 組投資組合，以大公司高動能減去大公司低動能的報酬率計算溢酬 (premuim)，小公司同上，兩溢酬平均數即為 UMD，即：
  $$\tt{UMD=\frac{(s2mom3 - s2mom1)+(s1mom3 - s1mom1)}{2}}$$
  
`補充`  
報酬動能 (Momentum) 的定義為：公司過去 12 個月到過去 2 個月的累積報酬，不同於上述因子，動能因子採每月重新分組，且動能因子的市值分組 (s1,s2) 也是每月分組，採上個月月底市值。  
詳細方法可參考 [Kenneth R. French 動能因子](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_mom_factor.html)，或文獻 [Carhart, M. M. (1997). On Persistence in Mutual Fund Performance. The Journal of Finance, 52(1), 57–82.](https://onlinelibrary.wiley.com/doi/10.1111/j.1540-6261.1997.tb03808.x)，這裡以 Kenneth R. French 網站的方式計算，即考慮規模效應。

---    
由於計算 Fama 因子與動能因子的投資組合再平衡時間不同，因此分開來計算。

# Fama–French Factor Calculation

首先載入套件、python 連結 MSSQL，準備取出在 MSSQL 中整理好的資料作運算。

In [1]:
import pandas as pd
import numpy as np
import urllib
from sqlalchemy import create_engine
# jupyter lab table 
import itables
itables.init_notebook_mode()
#
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=FinMind_DB;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

`1. df_pivot：`先把 MSSQL 中的財報`sql_pivot`從長表轉寬表,方便後續計算 factor variable。

In [2]:
sql_pivot = pd.read_sql_query(
    sql = '''
    SELECT 
        report_date,
        stock_id,
        item,
        item_value
    FROM silver.financial_statement
    ''',
    con = engine)
#
df_pivot = pd.pivot(
        sql_pivot,
        columns = 'item',
        index = ['report_date','stock_id'],
        values = 'item_value'
        ).reset_index()
# show sample data
df_pivot.astype({'report_date':'string'})[lambda x: x.report_date.str.contains('2020')].head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`2. df_variable：`根據財報的`report_date`和市值表的`month_date`JOIN (3,6,9,12月)市值，然後計算 factor variable (bm, op, inv)。

In [3]:
sql_mv = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        stock_id,
        market_value AS mv
    FROM silver.month_market_value
    ''',
    con = engine)
# 財報合併月底市值, 根據公司 Q1~Q4 財報合併 3,6,9,12 月市值
df_variable = pd.merge(
    left = df_pivot,
    right = sql_mv,
    how = 'left',
    left_on = ['report_date','stock_id'],
    right_on = ['month_date','stock_id'],
    )
# 篩選資料, 最後排序財報時間用於計算總資產成長率
df_variable = df_variable[
    lambda x: (x.TotalAssets>0) & (x.mv>0)
    ].sort_values(['report_date','stock_id'])
#
df_variable = df_variable.assign(
    bm = lambda x: x.Equity/x.mv,
    inv = lambda x: 
        (x.TotalAssets/x.groupby('stock_id').TotalAssets.shift(1)) -1,
    op = lambda x: 
        (x.Revenue-x.CostOfGoodsSold-x.OperatingExpenses-x.InterestExpense)/
        x.Equity
    )
# show sample data
df_variable.head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`3. df_group：`匯入公司產業表`silver.company_industry`把金融業與興櫃公司排除掉，接著排序 factor variable, 把個股分類進因子的排序分組中。

In [4]:
# 如公司的變數為 null, 把公司分配進 'NA' 組別, quantile() 不會計算到 null
sql_ci = pd.read_sql_query(
    sql = '''
    SELECT 
        market,
        industry_category,
        company_id
    FROM silver.company_industry
    ''',
    con = engine)
#
sr_id = sql_ci[
    lambda x: (x.market != 'emerging') & 
    (~x.industry_category.isin(['金融保險業','金融業']))
    ].company_id
#
df_group = df_variable[lambda x: x.stock_id.isin(sr_id)].assign(
    g_mv = lambda x: np.select(
        condlist = [
        x.mv < x.groupby('report_date').mv.transform(lambda y:y.quantile(0.5)),
        x.mv >= x.groupby('report_date').mv.transform(lambda y: y.quantile(0.5))
        ],
        choicelist = ['s1','s2'],
        default = 'NA'),
    g_bm = lambda x: np.select(
        condlist = [
        x.bm < x.groupby('report_date').bm.transform(lambda y:y.quantile(0.3)),
        x.bm < x.groupby('report_date').bm.transform(lambda y: y.quantile(0.7)),
        x.bm >= x.groupby('report_date').bm.transform(lambda y: y.quantile(0.7))
        ],
        choicelist = ['bm1','bm2','bm3'],
        default = 'NA'),
    g_op = lambda x: np.select(
        condlist = [
        x.op < x.groupby('report_date').op.transform(lambda y:y.quantile(0.3)),
        x.op < x.groupby('report_date').op.transform(lambda y: y.quantile(0.7)),
        x.op >= x.groupby('report_date').op.transform(lambda y: y.quantile(0.7))
        ],
        choicelist = ['op1','op2','op3'],
        default = 'NA'),
    g_inv = lambda x: np.select(
        condlist = [
        x.inv < x.groupby('report_date').inv.transform(lambda y:y.quantile(0.3)),
        x.inv < x.groupby('report_date').inv.transform(lambda y: y.quantile(0.7)),
        x.inv >= x.groupby('report_date').inv.transform(lambda y: y.quantile(0.7))
        ],
        choicelist = ['inv1','inv2','inv3'],
        default = 'NA')
    )[['report_date','stock_id','mv','g_mv','g_bm','g_op','g_inv']]
# show sample data
df_group.head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`4. df_vw：`組合市值與其他因子分組, 計算公司在所屬組別中的市值占比。

In [5]:
# 
df_group = df_group.assign(
    g_HML = lambda x: x.g_mv + '_' + x.g_bm,
    g_RMW = lambda x: x.g_mv + '_' + x.g_op,
    g_CMA = lambda x: x.g_mv + '_' + x.g_inv,
    )
# 根據公司的財報時間與所屬組別，計算各因子中公司的市值在組別中的佔比
df_vw = df_group.assign(
    HML_ratio = lambda x: x.mv/
    x.groupby(['report_date','g_HML']).mv.transform('sum'),
    RMW_ratio = lambda x: x.mv/
    x.groupby(['report_date','g_RMW']).mv.transform('sum'),
    CMA_ratio = lambda x: x.mv/
    x.groupby(['report_date','g_CMA']).mv.transform('sum'),
    )
# show sample data
df_vw.head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`5. df_vw_ret：`根據 `sql_date` 的日期對應合併 `df_vw`, 再把公司的月報酬`sql_return` JOIN 進來，依照公司在所屬組別中的`市值占比` × `公司月報酬`,計算公司在組別中的加權報酬率。

In [6]:
# sql_date 即上述 '因子計算方式' 提到的再平衡時間點, month_date 為月報酬的日期, report_date_by_quarter 為季財報的日期
sql_date = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        report_date_by_quarter
    FROM gold.dim_date
    ''',
    con = engine)
# show sample data
sql_date.head(36)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [7]:
sql_return = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        stock_id,
        month_return
    FROM silver.stock_month_return
    ''',
    con = engine)
#
df_vw_ret = pd.merge(
    left = df_vw, 
    right = sql_date,
    how = 'left',
    left_on = ['report_date'],
    right_on = ['report_date_by_quarter'],
    ).merge(
    sql_return,
    how = 'left',
    on = ['stock_id','month_date']
    ).assign(
    HML_vwr = lambda x: x.month_return*x.HML_ratio,
    RMW_vwr = lambda x: x.month_return*x.RMW_ratio,
    CMA_vwr = lambda x: x.month_return*x.CMA_ratio
        )
# show sample data
df_vw_ret.head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`6. df_port:`最後根據每個日期與因子分組, 加總公司的加權報酬率，完成因子投資組合的加權報酬率計算。

In [8]:
# 不同因子分開計算, 再 pd.concat 成一個長表, 3 個 column 分別命名為 'month_date','port','return', 去除 NA 投組
df_port = pd.concat(
    [
    df_vw_ret.groupby(['month_date','g_HML']).agg(
    {'HML_vwr':'sum'}).reset_index().set_axis(
    ['month_date','port','return'], axis = 1),
        
    df_vw_ret.groupby(['month_date','g_RMW']).agg(
    {'RMW_vwr':'sum'}).reset_index().set_axis(
    ['month_date','port','return'], axis = 1),
        
    df_vw_ret.groupby(['month_date','g_CMA']).agg(
    {'CMA_vwr':'sum'}).reset_index().set_axis(
    ['month_date','port','return'], axis = 1),
    ],
    axis = 0).astype({'month_date':'string'})[
    lambda x: (~x.port.str.contains('NA')) & 
              (x.month_date <= '2026-05-01')
    ]
# show all data
df_port

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


> `SQL Data Validation:`使用 MSSQL 計算的資料驗證上述 python 計算的資料。

In [9]:
# read MSSQL 每季在平衡_因子投組
sql_port_q = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        portfolio,
        portfolio_return 
    FROM gold.fct_factor_portfolio_rebalance_quarterly
    ''',
    con = engine).astype({'month_date':'string'}).set_axis(
    ['month_date','port','return'],axis = 1)

# 合併 python 計算的因子投組, 兩資料報酬率相減後取絕對值, 比較計算結果的報酬率差距
df_abs = sql_port_q.merge(
    df_port,
    how = 'left',
    on = ['month_date','port']).assign(
    ret_diff = lambda x: 
        (x.return_x - x.return_y).abs()
    )
# 排除報酬率一樣的計算結果, 由大到小排序 報酬率差距 的絕對值, 對比計算結果
df_abs[lambda x: x.ret_diff > 0].sort_values('ret_diff',ascending = False) 

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


報酬率差距最大的投組是`2026/04 – s2_bm1`，但差距約為小數點後 16 位數，差距來源很明顯是浮點數運算，故上述 python 與 MSSQL 的計算結果無差異。

# Momentum Factor Calculation

`1. df_mom:`載入月報酬資料`sql_return`，計算各公司的 mom。

In [2]:
#
sql_return = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        stock_id,
        month_return
    FROM silver.stock_month_return
    ''',
    con = engine)
# 計算動能因子變數: mom, 公司 t 月 mom 為 t-12 至 t-2 月的累積報酬
df_mom = sql_return.sort_values(['month_date','stock_id']).assign(
    mom = lambda x: x.groupby('stock_id').month_return.transform(
        lambda y: pd.Series(
            [1+y.shift(i) for i in range(2,13)]).prod())
    )
# show sample data
df_mom.sort_values(['stock_id','month_date']).head(24)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`2. df_mom_mv:` JOIN 公司月市值`sql_mv`再把市值 lag 1 個月, 用於後續計算市值加權和分組。

In [3]:
sql_mv = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        stock_id,
        market_value
    FROM silver.month_market_value
    ''',
    con = engine)
# 
df_mom_mv = df_mom.merge(
    sql_mv,
    how = 'left',
    on = ['month_date','stock_id'])
# lag 市值前先排序資料日期, 在根據公司分別去 shift 市值
df_mom_mv = df_mom_mv.sort_values(['month_date']).assign(
    lag_mv = lambda x: x.groupby('stock_id').market_value.shift(1)
    )
# show sample data
df_mom_mv.sort_values(['stock_id','month_date']).dropna(subset = 'market_value').head(24)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`3. df_group:`每個月根據 mom & lag_mv 把公司分類進組別中,再組合成 UMD 的投資組合。

In [4]:
df_group = df_mom_mv.assign(
    g_mv = lambda x: np.select(
        condlist = [
        x.lag_mv < x.groupby('month_date').lag_mv.transform(lambda y:y.quantile(0.5)),
        x.lag_mv >= x.groupby('month_date').lag_mv.transform(lambda y: y.quantile(0.5))
        ],
        choicelist = ['s1','s2'],
        default = 'NA'),
    g_mom = lambda x: np.select(
        condlist = [
        x.mom < x.groupby('month_date').mom.transform(lambda y:y.quantile(0.3)),
        x.mom < x.groupby('month_date').mom.transform(lambda y: y.quantile(0.7)),
        x.mom >= x.groupby('month_date').mom.transform(lambda y: y.quantile(0.7))
        ],
        choicelist = ['mom1','mom2','mom3'],
        default = 'NA'),
    g_UMD = lambda x: x.g_mv + '_' + x.g_mom,
    )
# show sample data
df_group.dropna().head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`4. df_vw_ret:`計算公司在所屬組別中的市值占比，再以公司`市值占比×公司月報酬`, 計算出公司市值加權月報酬率。

In [5]:
df_vw_ret = df_group.assign(
    UMD_ratio = lambda x: x.lag_mv/
    x.groupby(['month_date','g_UMD']).lag_mv.transform('sum'),
    UMD_vwr = lambda x: x.month_return*x.UMD_ratio
    )
# show sample data
df_vw_ret.dropna().head()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


`5. df_port:`根據日期與投資組合加總公司市值加權報酬率，完成動能因子投組市值加權報酬率。

In [6]:
df_port = df_vw_ret.groupby(['month_date','g_UMD']).agg(
    {'UMD_vwr':'sum'}).reset_index().set_axis(
    ['month_date','port','return'], axis = 1)
# 去除 NA 投組
df_port = df_port.astype({'month_date':'string'})[
    lambda x: 
        (~x.port.str.contains('NA')) & 
        (x.month_date <= '2026-05-01')
    ]
#
df_port

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


>`SQL Data Validation:`使用 MSSQL 計算的資料驗證上述 python 計算的資料。

In [8]:
# read MSSQL 動能因子投組
sql_port_mom = pd.read_sql_query(
    sql = '''
    SELECT 
        month_date,
        portfolio,
        portfolio_return 
    FROM gold.fct_mom_portfolio
    ''',
    con = engine).astype({'month_date':'string'}).set_axis(
    ['month_date','port','return'],axis = 1)
# 合併 python 計算的動能投組, 兩資料報酬率相減後取絕對值, 比較計算結果的報酬率差距
df_abs = sql_port_mom.merge(
    df_port,
    how = 'left',
    on = ['month_date','port']).assign(
    ret_diff = lambda x: 
        (x.return_x - x.return_y).abs()
    )
# 排除報酬率一樣的計算結果, 由大到小排序 報酬率差距 的絕對值, 對比計算結果
df_abs[lambda x: x.ret_diff > 0].sort_values('ret_diff', ascending = False)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


動能因子的投組同 Fama 投組，報酬率最大差距的絕對值約為小數點後 16 位數，也是只有浮點數運算上的差異而已。

-----------

# Tableau 資料前處理

如果是計算每年再平衡的因子投組，只需在一開始把 MSSQL 財報資料`silver.financial_statement`整理成年財報後，在合併`gold.dim_date`時改合併年財報的日期即可，詳細內容可查看`ddl_year_factor_portfolio.sql`檔案。  
Tableau Public 是使用 excel 檔，故最後使用 python 把 MSSQL 資料取出來再轉成 excel 檔，Dashboard 總共有使用到`gold.fct_factor_portfolio_rebalance_quarterly`、`gold.fct_factor_portfolio_rebalance_annual`、`gold.fct_mom_portfolio`、`gold.fct_market_premium`(加權指數報酬率和無風險利率)、`gold.fct_factor_data`(因子投組計算的因子報酬率資料)。

In [ ]:
# 所有資料整理成一個長表, 補上 'rebalance' column
df_portfolio = pd.read_sql_query(
    sql =
    '''
    SELECT 
         month_date,
         'quarterly' rebalance,
         portfolio,
         portfolio_return
    FROM gold.fct_factor_portfolio_rebalance_quarterly
    UNION ALL 
    SELECT 
         month_date,
         'annual' rebalance,
         portfolio,
         portfolio_return
    FROM gold.fct_factor_portfolio_rebalance_annual
    UNION ALL 
    SELECT 
         month_date,
         'monthly' rebalance,
         portfolio,
         portfolio_return
    FROM gold.fct_mom_portfolio
    UNION ALL 
    SELECT 
         month_date,
         'monthly' rebalance,
         portfolio,
         portfolio_return
    FROM gold.fct_market_premium
    UNION ALL 
    SELECT 
         month_date,
         rebalance,
         portfolio,
         portfolio_return
    FROM gold.fct_factor_data
    ''',
    con = engine).astype({'month_date':'string'})
# 去除 CMA 的 null 跟 加權指數報酬率的第一筆資料, 也是 null
df_portfolio = df_portfolio[
    lambda x: x.month_date <= '2026-05-01'].dropna()
# 匯出 excel 檔
# df_portfolio.to_excel(
#     'factor_portfolio.xlsx',
#     index = False,
#     sheet_name = 'portfolio')